# A40 12-hour campaign — live monitor

Training runs in tmux session `a40-12h`, independently of this notebook. Closing Jupyter or disconnecting from RunPod does not stop training while the pod remains running.

The deadline is **2026-09-17 15:15:44 UTC**. Loss and CelebA preprocessing are unchanged. The controller screens six configurations: two ADM U-Net widths (23.0M and 51.8M parameters), each at learning rates 1e-4, 2e-4, and 3e-4. Each pilot receives 25 minutes, crediting existing training time. The best learning rate per width receives another 45 minutes; the stronger finalist then trains for the remaining budget, roughly six hours after evaluation overhead. The final 60 minutes are reserved for FID evaluation, within the same fixed deadline.

Run the next cell whenever you reconnect. The final cell refreshes automatically; interrupting it only stops the monitor.

In [ ]:
from pathlib import Path
import json, time
from datetime import datetime, timezone
from IPython.display import display, Markdown, Image, clear_output
ROOT = Path('/workspace/Diffusion')
CAMPAIGN = ROOT / 'outputs/campaign_20260917'

def show_status():
    status_path = CAMPAIGN / 'status.json'
    if not status_path.exists():
        print('Controller has not written status yet.')
        return
    status = json.loads(status_path.read_text())
    remaining = max(0, status['deadline_unix'] - time.time())
    display(Markdown(f"**Phase:** {status['phase']}  |  **Budget remaining:** {remaining/3600:.2f} hours"))
    for name, candidate in status.get('candidates', {}).items():
        directory = Path(candidate['run_dir'])
        info_path = directory / 'run.json'
        info = json.loads(info_path.read_text()) if info_path.exists() else {}
        url = info.get('wandb_url')
        display(Markdown(f"**{name}**: {candidate['benchmark']['parameters']:,} parameters" + (f" — [W&B]({url})" if url else '')))
        previews = sorted((directory / 'previews').glob('*.png'), key=lambda path: path.stat().st_mtime)
        if previews:
            print(previews[-1].name)
            display(Image(filename=str(previews[-1]), width=1000))
    if status.get('evaluations'):
        print('FID evaluations (compare matching sample counts and solver steps):')
        for result in status['evaluations']:
            print(result['name'], 'step', result.get('checkpoint_step'), 'n', result['num_generated'], 'Euler', result['ode_steps'], 'FID', round(result['fid'], 3))
    if status.get('selected_model'):
        display(Markdown('**Selected model**'))
        print(json.dumps(status['selected_model'], indent=2))
    logs = sorted((CAMPAIGN / 'logs').glob('*.log'), key=lambda path: path.stat().st_mtime)
    if logs:
        with logs[-1].open('rb') as stream:
            stream.seek(max(0, logs[-1].stat().st_size - 1800))
            print(stream.read().decode(errors='replace').replace('\r', '\n')[-1400:])
    return status

show_status()

In [ ]:
# Safe to interrupt: this loop only reads progress and displays existing images.
while True:
    clear_output(wait=True)
    status = show_status()
    if status and status.get('phase') in {'complete', 'failed'}:
        break
    time.sleep(15)